# SMS Spam Collection — Data Cleaning with Pandas

**Author:** Sarthak Rote  
**Dataset:** UCI SMS Spam Collection (5,574 messages)  
**Goal:** Download, inspect, clean duplicate/missing records, count `ham` and `spam`, and export exactly two columns: `label` and `message`.

Dataset source: https://archive.ics.uci.edu/dataset/228/sms%2Bspam%2Bcollection  
Citation: Almeida, T. & Hidalgo, J. (2011). SMS Spam Collection. UCI Machine Learning Repository. DOI: 10.24432/C5CC84. License: CC BY 4.0.

## 1. Import libraries

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

import pandas as pd

## 2. Download and extract the dataset

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
ZIP_PATH = Path("sms_spam_collection.zip")
DATA_PATH = Path("SMSSpamCollection")

if not DATA_PATH.exists():
    print("Downloading the official UCI dataset...")
    with urllib.request.urlopen(DATA_URL, timeout=60) as response:
        ZIP_PATH.write_bytes(response.read())

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        # Extract only the original tab-separated message file.
        DATA_PATH.write_bytes(archive.read("SMSSpamCollection"))
else:
    print("Dataset file already exists. Skipping download.")

print("Dataset ready:", DATA_PATH.resolve())

## 3. Load data using Pandas

In [ ]:
df = pd.read_csv(
    DATA_PATH,
    sep="\\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8"
)

print("Dataset loaded successfully.")
df.head()

## 4. Explore rows, columns and labels

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\\nColumn names:", df.columns.tolist())
print("\\nData types:")
print(df.dtypes)
print("\\nFirst five records:")
display(df.head())
print("\\nLabel values:")
print(df["label"].value_counts(dropna=False))

## 5. Check missing values

In [ ]:
print("Missing values in each column:")
print(df.isna().sum())

print("\\nTotal rows containing at least one missing value:")
print(df.isna().any(axis=1).sum())

## 6. Remove missing and duplicate records

In [ ]:
before_rows = len(df)

# Remove records with a missing label or message.
df = df.dropna(subset=["label", "message"]).copy()

# Remove accidental leading/trailing spaces.
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str).str.strip()

# Keep only valid target labels.
df = df[df["label"].isin(["ham", "spam"])].copy()

after_missing = len(df)
print("Rows before cleaning:", before_rows)
print("Rows after missing/invalid-value cleaning:", after_missing)
print("Missing/invalid rows removed:", before_rows - after_missing)

duplicate_count = df.duplicated(subset=["label", "message"]).sum()
print("\\nDuplicate rows found:", duplicate_count)

df = df.drop_duplicates(subset=["label", "message"]).reset_index(drop=True)
print("Rows after duplicate removal:", len(df))

## 7. Count ham and spam messages

In [ ]:
label_counts = df["label"].value_counts()
print("Message counts:")
print(label_counts)

print("\\nPercentages:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

## 8. Keep exactly the required two columns

In [ ]:
cleaned_df = df[["label", "message"]].copy()

print("Final shape:", cleaned_df.shape)
print("Final columns:", cleaned_df.columns.tolist())
display(cleaned_df.head())

## 9. Save final cleaned dataset

In [ ]:
OUTPUT_PATH = "cleaned_sms.csv"
cleaned_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Saved successfully: {OUTPUT_PATH}")
print("Final file columns:", pd.read_csv(OUTPUT_PATH, nrows=0).columns.tolist())

## 10. Final verification

In [ ]:
check_df = pd.read_csv(OUTPUT_PATH)

assert check_df.columns.tolist() == ["label", "message"]
assert check_df.isna().sum().sum() == 0
assert not check_df.duplicated(subset=["label", "message"]).any()
assert set(check_df["label"].unique()).issubset({"ham", "spam"})

print("All checks passed!")
print("Final rows:", len(check_df))
print("Ham:", (check_df["label"] == "ham").sum())
print("Spam:", (check_df["label"] == "spam").sum())
print("Columns:", check_df.columns.tolist())

## Conclusion

- The official UCI SMS Spam Collection was downloaded and loaded with Pandas.
- Rows, columns, data types, labels and missing values were inspected.
- Missing/invalid rows and duplicate records were removed.
- Ham and spam counts were calculated.
- The final CSV contains only `label` and `message`.

**Note:** Run all notebook cells from top to bottom while connected to the internet. The official UCI dataset download is performed in Step 2. The notebook creates `cleaned_sms.csv` in the same folder.